# 11. Training dan generasi halaman utuh

Loss teacher forcing, raw generation, review manusia dan pemeriksaan browser adalah bukti yang berbeda. Final hanya dibuka setelah kandidat terkunci; notebook pengembangan ini tidak membuka final.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
while not (ROOT / '.venv').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/fullpage').is_dir(), 'Open using the project kernel and folder'
def read(path): return json.loads((ROOT / path).read_text(encoding='utf-8'))
display(Markdown('Environment proyek: ' + str(ROOT)))

In [ ]:
display(read('configs/fullpage/training-v2.json'))
for run in ('TRAIN-016','TRAIN-017'):
    summary = read(f'reports/training/{run}.json')
    display({k:summary.get(k) for k in ('run_id','status','global_step','best_validation_loss','wall_seconds','total_train_tokens','peak_allocated_bytes','free_bytes_after')})

## Kurva dan throughput aktual

Training utama berdiri sendiri dari INIT2048, budget2.400. Best dipilih dengan ordinary CE penuh pada development. Best disimpan weights-only; exact resume memakai last.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
for run in ('TRAIN-016','TRAIN-017'):
    metrics = pd.read_json(ROOT / f'outputs/metrics/{run}.jsonl',lines=True)
    train = metrics[metrics.event=='train']; val = metrics[metrics.event=='validation']
    axes[0].plot(val.step,val.loss,'o-',label=run+' validation')
    axes[1].plot(train.step,train.tokens_per_second,label=run+' target tokens/s')
    print(run, 'target tokens:',int(train.valid_tokens.sum()),'long-position labels:',int(train.supervised_tokens_after_position_1024.sum()),'mean target tokens/s:',int(train.valid_tokens.sum()/train.seconds.sum()))
axes[0].set(xlabel='Update',ylabel='Ordinary CE',yscale='log'); axes[1].set(xlabel='Update',ylabel='Supervised tokens/s')
for ax in axes: ax.legend()
plt.show()

In [ ]:
for run in ('TRAIN-016','TRAIN-017'):
    gpu = pd.read_json(ROOT / f'outputs/metrics/{run}_gpu.jsonl',lines=True)
    display(gpu[['utilization_pct','memory_used_mib','power_w','temperature_c']].agg(['mean','max']))
    task_report = ROOT / f'reports/training/{run}_TASK_LOSS.json'
    if task_report.exists(): display(read(str(task_report.relative_to(ROOT))))

## Generation pilot dan development

Plan512, section1536; output yang tidak valid gagal, tidak diperbaiki memakai target. Retry maksimum sekali. Halaman diterima hanya jika semua fakta dan komponen lolos.

In [ ]:
for filename in ('TRAIN-016_PILOT.json','TRAIN-017_DEVELOPMENT_BEST.json','TRAIN-017_DEVELOPMENT_LAST.json'):
    path = ROOT / 'reports/fullpage' / filename
    if path.exists():
        report = read(str(path.relative_to(ROOT)))
        display({k:report.get(k) for k in ('scope','complete','total','first_attempt_complete','after_retry_complete','model_gate','groups','failures','page_seconds','browser_status','human_review_status')})

In [ ]:
directory = ROOT / 'outputs/fullpage/TRAIN-017-development-best'
if directory.exists():
    paths = sorted(directory.glob('*/attempt-*/result.json'))
    accepted = [p for p in paths if json.loads(p.read_text(encoding='utf-8'))['status']=='accepted']
    selected = accepted[0] if accepted else paths[0]
    result = json.loads(selected.read_text(encoding='utf-8'))
    print('Raw model example:',result['request_id'],result['status'])
    display(pd.DataFrame([{k:a.get(k) for k in ('stage','retry','stop_reason','generated_tokens','prefix_tokens','seconds','errors')} for a in result['attempts']]))
    print('Raw plan:',result['attempts'][0].get('raw','')[:1800])
    plan_path = selected.parent / 'plan.json'
    if plan_path.exists(): display(json.loads(plan_path.read_text(encoding='utf-8')))
    website = selected.parent / 'website/page_spec.json'
    if website.exists(): display(json.loads(website.read_text(encoding='utf-8')))

## Status akhir dan opt-in training

UI tetap tahap akhir. Jangan menyatakan model siap jika gate development belum95%, review manusia belum diisi, atau browser belum diperiksa. Sel di bawah tidak memulai training ketika notebook dibuka.

In [ ]:
progress = ROOT / 'reports/fullpage/AI-018_PROGRESS.json'
if progress.exists():
    snapshot = read(str(progress.relative_to(ROOT)))
    display({k:v for k,v in snapshot.items() if k!='evidence'})
RUN_TRAINING = False
if RUN_TRAINING:
    raise RuntimeError('Create a new logged run ID or exact-resume from last; preserve previous artifacts. Use qa.run_logged and qa.train_fullpage.')
print('Training automatic execution:',RUN_TRAINING)